# Kampala Taxi Route Revenue, Pricing & Fleet Planner

The OOP classes are in `src/taxi.py`; analysis, backtesting, equilibrium, fleet planning, and seasonal simulation are in `src/taxi_analysis.py`. The notebook imports and uses those modules.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.taxi import (
    ExponentialSmoothingForecaster,
    LinearTrendForecaster,
    MovingAverageForecaster,
    Route,
    SeasonalNaiveForecaster,
)
from src.taxi_analysis import (
    create_routes,
    run_analysis,
    solve_ntinda_equilibrium,
)

## Ntinda supply–demand equilibrium

Rearrange $Q_d=120-0.02P$ and $Q_s=10+0.03P$, then set $Q_d=Q_s=Q$. This gives $Q+0.02P=120$ and $Q-0.03P=10$, solved as a 2×2 system. The model equilibrium is not a recommendation to change fares without considering service quality, regulation, affordability, and the fact that the equations are assumptions.

In [ ]:
equilibrium = solve_ntinda_equilibrium()
print('System matrix:')
print(equilibrium['coefficients'])
print('Right-hand side:', equilibrium['right_hand_side'])
print(f"Equilibrium fare: UGX {equilibrium['equilibrium_fare_ugx']:,.0f}")
print(f"Equilibrium quantity: {equilibrium['equilibrium_quantity_passengers_per_trip_hour']:.0f} passengers/trip-hour")
print(f"Current fare is {equilibrium['current_fare_position']} equilibrium; modeled current-fare shortage: {equilibrium['shortage_at_current_fare']:.0f} passengers/trip-hour")

In [ ]:
results = run_analysis()
results['route_statistics'].round(2).to_string()

## Walk-forward validation

For each route, each test prediction uses only observations available before that day; origins cover days 4–10. This rolling forecasting-origin approach avoids using future observations in each fit [2]. Simple exponential smoothing selects α independently per route by the lowest rolling-origin MAE across α values from 0.1 to 0.9; SES weights recent observations more heavily as α increases [1]. The model with the lowest backtest MAE is refit on all ten days for the day-11 forecast. Seven validation days are a small sample, so rankings can change with more history.

In [ ]:
print('Rolling-origin MAE, passengers/day:')
print(results['model_mae'].pivot(
    index='route', columns='model', values='MAE_passengers'
).round(2).to_string())

alpha_rows = [
    {'route': route_name, 'selected_alpha': evaluation['ses_alpha'],
     **{f'alpha_{alpha:.1f}': score for alpha, score in evaluation['ses_alpha_grid_mae'].items()}}
    for route_name, evaluation in results['route_evaluations'].items()
]
print('SES alpha grid MAE by route:')
print(pd.DataFrame(alpha_rows).set_index('route').round(2).to_string())

In [ ]:
fleet_rows = [
    {'route': route, **forecast}
    for route, forecast in results['day_11_forecasts'].items()
]
fleet_table = pd.DataFrame(fleet_rows).set_index('route')
print(fleet_table[[
    'selected_model', 'forecast_passengers', 'fare_ugx',
    'forecast_revenue_ugx', 'buffered_passengers',
    'capacity_per_vehicle_passengers', 'vehicles',
]].round(2).to_string())

In [ ]:
revenue_figure, revenue_axis = plt.subplots(figsize=(12, 7))
revenue_axis.imshow(plt.imread(results['revenue_plot_path']))
revenue_axis.axis('off')
revenue_figure

## Weekly seasonality extension

The seeded 60-day extension applies route-specific base demand, a weekday pattern (Friday higher, Sunday lower), and modest noise. Seasonal-naïve forecasts use the observation from seven days earlier, the standard same-season benchmark [1]. This should outperform a 3-day average when weekly repetition is strong, but the generated pattern is synthetic, not evidence that actual passenger demand follows this exact pattern.

In [ ]:
seasonal = results['seasonal_extension']
print(seasonal['comparison'].pivot(
    index='route', columns='model', values='MAE'
).round(2).to_string())
print('Mean MAE:', seasonal['mean_mae_by_model'])
print('Seasonal naive wins:', seasonal['seasonal_naive_beats_moving_average'])

In [ ]:
seasonal_figure, seasonal_axis = plt.subplots(figsize=(12, 6))
seasonal_axis.imshow(plt.imread(results['seasonal_plot_path']))
seasonal_axis.axis('off')
seasonal_figure

## Findings & Limitations

Simple exponential smoothing with α=0.9 has the lowest rolling-origin MAE for all three routes: 5.88 passengers for Ntinda, 4.73 for Entebbe, and 3.79 for Mukono. The consistently high α means the selected forecasts respond strongly to the latest observation; with only seven validation days, this ranking is preliminary and may be sensitive to one unusual day. Entebbe’s fare makes its projected day-11 revenue the highest (about UGX 321,137), even though its passenger forecast is only about 64. Forecasting passenger counts separately from fare also assumes fares and route mix remain unchanged.

The 15% buffer raises expected demand, but each route still rounds to one vehicle because one vehicle offers 112 passenger places per day under the eight-trip assumption. Fleet counts therefore change in steps, and one vehicle may be inadequate if trips are unevenly loaded, vehicles are unavailable, or passengers are concentrated at peak times. The Ntinda equilibrium fare of UGX 2,200 follows directly from the supplied linear curves; the implied 10-passenger shortage at UGX 2,000 is not a causal estimate or fare recommendation without observed supply, demand, and regulatory data.

Finally, seasonal-naïve MAE (2.65) beats the 3-day average (8.71) on deliberately weekly synthetic data. This demonstrates the method’s advantage when weekly repetition is real, not that the generated Friday/Sunday pattern describes actual riders. Collect longer, dated passenger counts by trip and time before using these results for staffing or pricing decisions.

## References

1. Hyndman, R. J., & Athanasopoulos, G. *Forecasting: Principles and Practice* (3rd ed.), §5.2 “Some simple forecasting methods” and §8.1 “Simple exponential smoothing.” OTexts. https://otexts.com/fpp3/simple-methods.html and https://otexts.com/fpp3/ses.html. (Moving/naïve methods, seasonal-naïve forecasts, and SES weighting.)
2. Hyndman, R. J., & Athanasopoulos, G. *Forecasting: Principles and Practice* (3rd ed.), §5.10 “Time series cross-validation.” OTexts. https://otexts.com/fpp3/tscv.html. (Rolling forecasting origins.)

Route passenger counts, fares, supply/demand curves, and vehicle-capacity assumptions are supplied by the assignment. Weekly extension values are generated synthetically and are not observed operator records.